# GE 是什么 — 全栈定位与核心概念

本节是 GE 图引擎开发的基础章节。学习 GE，首先需要了解它在昇腾全栈中的**定位**、**核心职责**、**加速原理**以及典型的**使用路径**。本节按如下结构展开：

- 昇腾全栈中的 GE（中间层定位 + 术语表）
- GE 的两大职责：图编译 与 图执行
- GE 的图编译加速技术
- 图编译的四个阶段（图准备 / 图拆分 / 图优化 / 图编译）
- 两条使用路径：离线 ATC+ACL 与 在线 GeSession
- 框架驱动路径（PyTorch/TorchAir、TF Adapter、MindSpore、PaddlePaddle）
- 离线与在线的典型应用场景
- 课后练习

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>本节聚焦"是什么"与"为什么快"，建立全局认知；动态 shape、下沉调度的工程细节将在后续章节深入。</p>
</blockquote>
</div>

## 1. 昇腾全栈中的 GE

为便于理解后续内容，学习本课程前请先了解如下术语和相关概念。

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>概念</th><th>全称</th><th>解释说明</th></tr>
</thead>
<tbody>
<tr><td>GE</td><td>Graph Engine</td><td>图引擎，昇腾平台计算图编译和运行的控制中心</td></tr>
<tr><td>CANN</td><td>Compute Architecture for Neural Networks</td><td>昇腾异构计算架构，面向 AI 场景的软硬件全栈</td></tr>
<tr><td>AscendIR</td><td>Ascend Intermediate Representation</td><td>昇腾中间表示，GE 编译流程使用的核心 IR</td></tr>
<tr><td>ATC</td><td>Ascend Tensor Compiler</td><td>昇腾张量编译器，GE 的离线编译工具</td></tr>
<tr><td>GeModel</td><td>Graph Engine Model</td><td>GE 编译后的内部模型对象，在线/离线路径的内部编译产物</td></tr>
<tr><td>OM</td><td>Offline Model</td><td>离线模型文件，离线编译场景中由 GeModel 序列化/落盘得到的部署产物</td></tr>
<tr><td>ACL</td><td>Ascend Computing Language</td><td>昇腾计算语言，面向昇腾芯片的运行时 API</td></tr>
<tr><td>GeSession</td><td>Graph Engine Session</td><td>GE 图引擎会话，在线构图编译执行的入口</td></tr>
<tr><td>Runtime</td><td>Ascend Runtime</td><td>昇腾运行时，向上承接 GE 下发的 stream/task，向下驱动硬件</td></tr>
<tr><td>NPU</td><td>Neural Network Processing Unit</td><td>神经网络处理单元，昇腾 AI 计算任务的主要执行设备</td></tr>
<tr><td>AI Core</td><td></td><td>NPU 内执行矩阵/向量等密集计算的核心，承载大部分算子</td></tr>
<tr><td>AI CPU</td><td></td><td>NPU 内执行控制类、标量类等不适合 AI Core 的算子的单元</td></tr>
<tr><td>Host</td><td></td><td>指与 Device 相连接的 x86 或 arm 服务器</td></tr>
<tr><td>Device</td><td></td><td>指集成了计算芯片的硬件设备，通过 PCIe 接口与 Host 连接</td></tr>
<tr><td>TorchAir</td><td>Torch Ascend Intermediate Representation</td><td>PyTorch 图模式扩展库，将 PyTorch FX 图转换为 AscendIR</td></tr>
<tr><td>TF Adapter</td><td>TensorFlow Adapter</td><td>TensorFlow 框架适配组件，将 TF GraphDef 转换为 AscendIR</td></tr>
</tbody>
</table>
</div>

在 CANN 异构计算架构中，GE 是连接 AI 框架与昇腾芯片的**图编译器 + 执行器**，位于整个软件栈的**中间层**：

- **向上**对接主流 AI 框架（PyTorch、TensorFlow、MindSpore、PaddlePaddle 等），将它们的算法模型统一转换为以 AscendIR 表示的计算图（Ascend Graph），并提供统一的图开发接口（GE API）支持自定义图结构；
- **向下**通过 Runtime 驱动昇腾硬件（AI Core / AI CPU 等）执行计算任务。

GE 的逻辑架构如下（上层框架 → GE Core[图编译器 / 图优化器 / 图执行器] + GE API → Runtime → Hardware）：

<p align="left"><img src="./images/ge_architecture.svg" alt="GE 逻辑架构图" width="70%"></p>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>GE 并不直接面向终端用户的应用代码，而是通过统一的图开发接口，既支持上层框架透明对接，也支持用户自定义构图与业务部署。</p>
</blockquote>
</div>

## 2. GE 的两大职责

GE 作为昇腾平台的图引擎，承担两大核心职责：**图编译** 与 **图执行**。

### 2.1 图编译

**将以 AscendIR 表示的计算图编译为昇腾可执行的 GeModel。**

在线/离线编译在 GE 内部形成的模型产物都是 **GeModel**。只有离线编译场景会进一步把 GeModel 序列化并落盘为 **OM（Offline Model）** 文件，用于独立部署。

GE Compiler 是 GE 的核心组件，其主要工作依次包括：

- **图级优化（Graph Optimization）**：在 AscendIR 层面进行通用编译器优化、算子融合等处理
- **算子编译**：结合图上推导得到的 shape 信息，对算子进行在线编译；若已有匹配的算子二进制，也可以复用二进制以减少编译耗时
- **流分配（Stream Planning）**：识别图中的可并行关系，将可并发的算子分配到不同流上
- **内存分配（Memory Planning）**：静态 shape 或动态分档场景可在编译/加载阶段依据确定 shape 做内存规划或分配；纯动态 shape 场景更多在执行时确定 shape/tiling，并申请或调整运行内存
- **模型序列化（Model Serialization）**：离线编译场景可选/离线路径特有，将编译好的 GeModel 序列化为 OM 文件

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>第 4 节会把图编译细化为"图准备 / 图拆分 / 图优化 / 图编译"四个阶段。</p>
</blockquote>
</div>

### 2.2 图执行

**在设备上加载和运行 GeModel；离线部署时，先从 OM 解析/恢复出 GeModel。**

GE Executor 负责模型在昇腾设备上的执行控制，包括：

- **模型加载（Model Loading）**：加载 GeModel，或在离线部署中解析 OM 并恢复 GeModel；静态 shape 场景通常在加载时申请真实内存、创建 stream 等运行资源，并将模型执行所需的算子二进制（bin）、权重等加载至 Device
- **模型执行（Model Execution）**：拷贝输入数据，将 stream/task 下发到 Device，由 AI Core/AI CPU 等执行对应算子；纯动态 shape 场景通常在执行期确定实际 shape/tiling，并申请或调整运行内存；计算完成后返回 Host

编译与执行的关系如下图所示：

<p align="left"><img src="./images/compile_execute_flow.svg" alt="GE 编译与执行关系图" width="82%"></p>

## 3. GE 的图编译加速技术

### 3.1 Eager 模式 vs 图模式

当前主流深度学习框架提供两种运行方式：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>模式</th><th>特点</th><th>局限 / 优势</th></tr>
</thead>
<tbody>
<tr><td><strong>Eager 模式</strong>（即时执行）</td><td>每个计算操作下发后<strong>立即执行</strong></td><td>调试直观，但只能看到单算子，缺乏全局视角，难以做整图优化</td></tr>
<tr><td><strong>图模式</strong></td><td>将所有计算操作构造成一张<strong>计算图</strong>，以图的形式下发执行</td><td>具备<strong>图的全局视角</strong>，可有效简化和优化整图，从而获得更优执行性能</td></tr>
</tbody>
</table>
</div>

GE 是专门面向图模式的 AI 编译器。相较于单算子依次下发的方式，图模式能够更有效地简化和优化计算图。

### 3.2 四大加速技术

GE 通过以下四类关键技术，减少 Host 与 Device 的调度交互、降低模型内存占用、提升整图执行性能：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>技术</th><th>作用</th></tr>
</thead>
<tbody>
<tr><td><strong>计算图优化</strong></td><td>在整图视角下做常量折叠、冗余消除、算子融合等，减少计算与搬运</td></tr>
<tr><td><strong>多流并行</strong></td><td>识别图中无依赖的算子，分配到不同 stream 上并发执行，提升硬件利用率</td></tr>
<tr><td><strong>内存复用</strong></td><td>静态 shape 或动态分档场景可按确定 shape 编排张量内存生命周期，复用空间，降低显存占用</td></tr>
<tr><td><strong>模型下沉</strong></td><td>整图一次性下发到 Device，执行时只下发一个 Task 触发，极大降低 Host 调度开销</td></tr>
</tbody>
</table>
</div>

### 3.3 模型下沉调度（重点）

AI 模型运行通常需要 Host（CPU，擅长复杂逻辑）与 Device（NPU，擅长高并行计算）协同。两种调度方式对比如下。

**① Host 调度（逐算子下发）**：Host 把模型中的算子**依次**下发到 Device 执行，每个算子在执行流上以 1 个或多个 Task 存在。问题在于：模型会运行多次，**每次运行都要把所有算子遍历下发一遍**，Host 与 Device 频繁交互成为瓶颈。

<p align="left"><img src="./images/host_schedule.svg" alt="Host 调度示意图" width="60%"></p>

**② 静态图下沉调度**：对于输入 shape 固定的静态 shape 模型，编译时即可确定所有算子的输入输出 shape，结合内存复用算法完成模型级内存编排，并提前完成 Tiling 等 Host 侧计算。动态分档场景在命中具体档位后，也可按该档位的确定 shape 做规划。于是 GE 提供下沉调度模式——

<p align="left"><img src="./images/model_sink.svg" alt="静态图下沉调度示意图" width="60%"></p>

其实现分两个阶段：

- **模型加载**：遍历图中所有算子，整体下发到 Device 流上，但**下发后不立即执行**。静态 shape 场景通常在加载时申请模型运行所需内存，这是一次性动作（首次执行时完成）。
- **模型执行**：加载完成后，像下发单算子 Task 一样，向执行流下发**一个"模型执行 Task"**；NPU 调度到该 Task 时，触发执行整图所有 Task。多次运行只需多次下发这一个 Task。纯动态 shape 场景通常在执行时确定实际 shape/tiling，并申请或调整运行内存。

每次下发模型时，支持刷新模型的 **Feature Map 内存地址**与**输入输出内存地址**；若地址发生更新，会在模型下沉头开销中完成图内算子相关地址的刷新。

**③ 时序对比**：模型下沉在执行开始有一个"模型下发"头开销，但 E2E 相对 Host 调度有收益；**头开销越小，性能提升幅度越大**。

<p align="left"><img src="./images/sink_timeline.svg" alt="下沉调度 Host/Device 时序对比" width="82%"></p>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>提示：动态 shape 场景下沉与上述细节将在<strong>第三章</strong>深入。另外，<strong>Atlas 350 加速卡不支持 UB 融合</strong>。</p>
</blockquote>
</div>

## 4. 图编译的四个阶段

对于以 AscendIR 表示的计算图，GE 适配底层硬件运行要求，进行一系列编译优化并生成 GeModel；离线编译场景可进一步把 GeModel 序列化/落盘为 OM。主要分为四个阶段：

<p align="left"><img src="./images/ge_compile_flow.svg" alt="图编译四阶段示意" width="85%"></p>

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>阶段</th><th>关键动作</th></tr>
</thead>
<tbody>
<tr><td><strong>① 图准备</strong></td><td>根据算子输入张量、算子逻辑及属性，推导输出张量的 <strong>shape / dtype / format</strong>（即 inferShapeAndType、inferFormat）。静态 shape 或动态分档场景可获得确定 shape，为后续内存规划/分配提供依据；纯动态 shape 场景则需要在执行期确定实际 shape/tiling。同时做<strong>与硬件无关、算法层级</strong>的优化：常量折叠、冗余分支消除等。</td></tr>
<tr><td><strong>② 图拆分</strong></td><td>按执行引擎（<strong>AI Core / AI CPU</strong> 等）对算子分类，拆分成不同子图，方便后续在不同硬件上进行针对性优化。</td></tr>
<tr><td><strong>③ 图优化</strong></td><td>通过算子融合等手段提升性能：<strong>硬件无关</strong>优化（如把多个算子融合成 1 个或几个，节省计算时间）；<strong>硬件相关</strong>优化（如通过 <strong>UB 融合</strong>缩短数据在硬件内存上的搬运时间）。<strong>Atlas 350 加速卡不支持 UB 融合。</strong></td></tr>
<tr><td><strong>④ 图编译</strong></td><td>根据计算图<strong>分配运行资源</strong>（内存规划/分配、stream 资源分配等）并生成 <strong>GeModel</strong>。静态 shape / 动态分档可在编译或加载阶段按确定 shape 做规划/分配；纯动态 shape 保留到执行期申请或调整运行内存。离线编译场景可选地将 GeModel 序列化为 <strong>.om</strong> 文件。</td></tr>
</tbody>
</table>
</div>

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>编译完成后进入<strong>图加载与图执行</strong>：图加载使用 GeModel；离线部署先解析 OM 并恢复 GeModel。静态 shape 场景加载时申请真实内存、创建 stream；纯动态 shape 场景更多在执行时确定 shape/tiling 并申请或调整运行内存；随后下发 stream/task 到 Device 执行并返回结果（见 <a href="../02_inference_workflow/02.02_offline_inference.ipynb">第 2.2 节 离线推理流程</a>）。</p>
</blockquote>
</div>

## 5. 两条使用路径

GE 面向不同的用户场景，提供了两条主要的使用路径：

### 5.1 离线路径：ATC + ACL

**ATC 编译模型文件 → GeModel → OM → ACL 推理部署**

在离线场景中，GE 不作为前端框架的后端参与执行流程，而是直接对模型文件进行编译：

- 用户提供已导出的模型（如 `.onnx`、`.pb`、`.air` 等）
- ATC（或 C++ Parser 接口）将其转换为 AscendIR，并调用 GE Compiler 生成 GeModel
- 离线编译路径将 GeModel 序列化/落盘为 OM 文件
- 使用 ACL API 在设备上加载 OM，解析/恢复为 GeModel 后执行推理

离线场景的特点是：
- **无需昇腾设备**（纯靠 Host 侧即可完成编译）
- **无需前端框架运行时**（不依赖 PyTorch/TF 图执行）
- **产物可独立部署**（OM 文件可在设备上加载，恢复为 GeModel 后执行）

### 5.2 在线路径：GeSession

**GeSession 实时构图编译执行**

在在线场景中，用户通过 GeSession API 在运行时构建 AscendIR 计算图，GE 实时完成编译和执行：

- 使用 GE API 构建 AscendIR 图
- 通过 GeSession 触发编译（CompileGraph），生成内部 GeModel
- 通过 GeSession 触发执行（RunGraph），加载并运行 GeModel

在线场景的特点是：
- **实时编译执行**（无需预先生成 OM 文件）
- **灵活可控**（支持动态构图、多图管理）
- **支持训练**（GeSession 支持梯度迭代）

## 6. 框架驱动路径

除了上述两条用户直接使用 GE 的路径外，GE 还被主流 AI 框架作为后端**透明驱动**。在这一模式下，GE 被框架直接调用、无需用户独立感知。

<p align="left"><img src="./images/torchair_arch.svg" alt="昇腾平台 PyTorch 图模式软件架构" width="28%"></p>

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>框架</th><th>适配组件</th><th>转换方式</th></tr>
</thead>
<tbody>
<tr><td>PyTorch</td><td>TorchAir</td><td>PyTorch FX 图 → AscendIR → GE 编译执行</td></tr>
<tr><td>TensorFlow 1.15</td><td>TF Adapter</td><td>TF GraphDef → AscendIR → GE 编译执行</td></tr>
<tr><td>TensorFlow 2.6.5</td><td>TF Adapter</td><td>tf.function 修饰的函数 → AscendIR → GE 编译执行</td></tr>
<tr><td>MindSpore</td><td>原生支持</td><td>MindSpore 原生对接 GE 图引擎</td></tr>
<tr><td>PaddlePaddle</td><td>官方适配</td><td>PaddlePaddle 对接 GE 图引擎</td></tr>
</tbody>
</table>
</div>

以 PyTorch 为例，TorchAir 作为 Ascend Extension for PyTorch（torch_npu）中支持图模式能力的扩展库，对接 PyTorch 的 Dynamo 特性，可将 PyTorch 的 FX 图转换为 AscendIR 表达的计算图，再通过 GE 进行编译优化并下发到昇腾硬件执行。

TorchAir 提供的昇腾编译后端可作为参数传入 `torch.compile` 接口，从而使能图模式执行：

```python
# 先导入 torch_npu，再导入 torchair
import torch
import torch_npu
import torchair

# 定义模型 Model
class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x, y):
        return torch.add(x, y)

# 实例化模型 model
model = Model()

# 从 TorchAir 获取 NPU 提供的默认 backend
config = torchair.CompilerConfig()
npu_backend = torchair.get_npu_backend(compiler_config=config)

# 使用 TorchAir 的 backend 调用 compile 接口编译模型
opt_model = torch.compile(model, backend=npu_backend)

# 执行编译后的 model
x = torch.randn(2, 2)
y = torch.randn(2, 2)
opt_model(x, y)
```

<div align="left">
<blockquote style="margin-left: 0; margin-right: auto;">
<p>框架驱动路径中，GE 作为 backend 透明运行，本课程不深入展开。</p>
</blockquote>
</div>

## 7. 离线与在线的典型应用场景

不同路径适用于不同场景，以下总结了常见的应用场景及其选择依据：

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>路径</th><th>典型场景</th><th>为什么适合</th></tr>
</thead>
<tbody>
<tr><td>离线：ATC + ACL</td><td>云端/边缘推理服务部署</td><td>模型结构固定，可提前编译 GeModel 并落盘为 OM，部署时由 ACL 加载执行</td></tr>
<tr><td>离线：ATC + ACL</td><td>视觉模型批量推理（ResNet、YOLO、OCR）</td><td>输入规格相对固定，适合静态 Shape 或动态分档编译</td></tr>
<tr><td>离线：ATC + ACL</td><td>大模型推理服务中的固定图部署</td><td>OM 可独立分发，统一打包权重、算子二进制等离线部署资源</td></tr>
<tr><td>离线：ATC + ACL</td><td>设备侧独立部署</td><td>不依赖前端训练框架运行时，只依赖 CANN/ACL 即可执行</td></tr>
<tr><td>在线：GeSession</td><td>框架适配路径（PyTorch/TorchAir、TF Adapter）</td><td>框架运行时驱动 GE 编译执行，用户不需要手动调用 ATC</td></tr>
<tr><td>在线：GeSession</td><td>训练或需要梯度迭代的场景</td><td>Session 可管理图生命周期并支持迭代执行</td></tr>
<tr><td>在线：GeSession</td><td>动态构图、动态图转静态图后的即时编译执行</td><td>图由程序实时构建，适合框架内部 backend 或实验性图执行</td></tr>
<tr><td>在线：GeSession</td><td>需要在一个进程中管理多张图的应用</td><td>单 Session 可添加、编译、运行、移除多张图</td></tr>
</tbody>
</table>
</div>

### 路径对比总结

<div align="left">
<table style="margin-left: 0; margin-right: auto;">
<thead>
<tr><th>维度</th><th>ATC + ACL（离线）</th><th>GeSession（在线）</th></tr>
</thead>
<tbody>
<tr><td>场景</td><td>模型部署推理</td><td>训练、在线推理、动态构图</td></tr>
<tr><td>编译时机</td><td>预编译，GeModel 可序列化为 OM 持久化</td><td>实时编译，Session 内完成</td></tr>
<tr><td>执行控制</td><td>ACL API，简单稳定</td><td>Session API，灵活可控</td></tr>
<tr><td>多图支持</td><td>每个离线 OM / GeModel 独立</td><td>单 Session 管理多图</td></tr>
<tr><td>训练能力</td><td>不支持</td><td>支持梯度迭代</td></tr>
</tbody>
</table>
</div>

## 课后练习

本节介绍了 GE 图引擎的全栈定位、核心职责、图编译加速技术与使用路径，请根据学习内容完成以下题目进行自测。

1. （判断题）GE 是昇腾平台计算图编译和运行的控制中心，提供了图构建、图编译优化及图执行控制等功能。

2. （判断题）在离线场景中，ATC 编译模型时需要昇腾设备在线才能完成编译。

3. （单选题）GE 的两大核心职责是什么？
    A. 模型训练和模型评估
    B. 图编译和图执行
    C. 数据预处理和数据增强
    D. 内存管理和网络通信

4. （单选题）以下哪个场景最适合使用离线路径（ATC + ACL）？
    A. 训练场景中需要梯度迭代
    B. 动态构图、图结构在运行时变化
    C. 云端推理服务部署，模型结构固定
    D. 需要在一个进程中管理多张图

5. （多选题）以下关于 GeSession 在线路径的描述，哪些是正确的？
    A. 支持实时构图、编译和执行
    B. 支持梯度迭代，可用于训练场景
    C. 单 Session 可管理多张图
    D. 需要预先生成 OM 文件才能执行

6. （单选题）GE 图编译的四个阶段中，"按执行引擎（AI Core / AI CPU 等）对算子分类，拆分形成不同子图"属于哪个阶段？
    A. 图准备
    B. 图拆分
    C. 图优化
    D. 图编译

7. （多选题）以下关于"图准备"阶段的描述，哪些是正确的？
    A. 通过 inferShapeAndType、inferFormat 推导张量的 shape/dtype/format
    B. 静态 shape 或动态分档场景可获得确定 shape，为后续内存规划/分配提供依据；纯动态 shape 通常需在执行期确定实际 shape/tiling
    C. 进行与硬件无关、算法层级的优化，如常量折叠、冗余分支消除
    D. 直接编译生成最终的 .om 离线模型，且在线场景也必须依赖 OM 执行

8. （判断题）在静态图下沉调度的"模型加载"阶段，整图算子会被下发到 Device 流上并立即开始执行。

9. （多选题）以下关于模型下沉调度的描述，哪些是正确的？
    A. 模型加载是一次性动作，遍历整图算子下发到 Device 但不立即执行
    B. 模型执行时只需向执行流下发一个"模型执行 Task"即可触发整图执行
    C. 相比 Host 逐算子调度，下沉调度可显著降低 Host 与 Device 的交互开销
    D. 每次模型下发都支持刷新 Feature Map 与输入输出内存地址

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.02_answer.txt